In [12]:
from IPython.core.display import publish_display_data
import pandas as pd

'''
gene_set= ["ANKRD22","BCL2L14","CXCL10","CXCL11","CXCL9","ETV7","GBP1","GBP5","GCH1","HAPLN3","IDO1","PLAAT4","STAT1","TAP1","TNFSF10","VAMP5","WARS1"]

prompt = f"""
I am assessing a single cell dataset of human macrophages. My data revealed a set of co-expressed genes:
{', '.join(gene_set)}
You are a biomedical expert. Based on your knowledge of gene expression patterns and protein function,
assess to which transcriptional program this list of genes most likely belongs.
If it is a technical artefact, or belongs to contaminating cells, indicate so.
Consider the gene set as a whole, and the cell population assayed. Do not assess each gene individually.

List the most relevant category defined by up to 3 words, e.g. "protein degradation", "S-phase", "antigen presentation", "technical artefact"
Rate your annotation certainty from 1 to 5, with 1 being an educated guess and 5 being certain.
Present your answer in this format:

Category:
Certainty:
Justification:
"""
from google.colab import ai

response = ai.generate_text(prompt, model_name='google/gemini-2.0-flash-lite')
print(response)


'''


text = response
lines = [l.strip() for l in text.splitlines()]

# ---- compact parsing ----
sections = {"category": [], "certainty": [], "justification": []}
current = None

for line in lines:
    if not line:
        continue
    lower = line.lower()
    if lower.startswith("category:"):
        current = "category"
        val = line.split(":", 1)[1].strip()
        if val:
            sections[current].append(val)
    elif lower.startswith("certainty:"):
        current = "certainty"
        val = line.split(":", 1)[1].strip()
        if val:
            sections[current].append(val)
    elif lower.startswith("justification:"):
        current = "justification"
        val = line.split(":", 1)[1].strip()
        if val:
            sections[current].append(val)
    else:
        if current is not None:
            sections[current].append(line)

# category: just join whatever was collected
raw_category = " ".join(sections["category"]).strip()
category = raw_category or None

# certainty: first digit 1–5 that appears
raw_cert = " ".join(sections["certainty"]).strip()
certainty = None
for ch in raw_cert:
    if ch.isdigit():
        certainty = int(ch)
        break

# justification: join all lines
raw_just = " ".join(sections["justification"]).strip()
justification = raw_just or None

df = pd.DataFrame([{
    "category": category,
    "certainty": certainty,
    "justification": justification,
}])


In [14]:
display (df)

,category,certainty,justification
0,Interferon Response,5,The gene set strongly indicates activation of ...
